# RTX 4050 Optimized VQA Fine-tuning (Target: 0.9+)

이 노트북은 RTX 4050(6GB VRAM) 환경에서 **Qwen2.5-VL-3B-Instruct** 모델을 최적화하여 학습하도록 설계되었습니다.

### 🛠 개선된 전략:
1. **Precise Label Masking**: Assistant의 응답 시작 위치를 정확히 계산하여 정답 토큰만 학습합니다.
2. **Logit-based Inference**: 생성된 텍스트 파싱 대신 'a', 'b', 'c', 'd' 토큰의 확률(Logits)을 직접 비교하여 정확도를 높입니다.
3. **Memory Optimization**: `gradient_checkpointing`과 4-bit 양자화를 사용하여 6GB VRAM에서도 안정적으로 학습합니다.
4. **Overfitting Control**: Dropout을 높이고 Weight Decay를 추가하여 Loss가 0이 되어도 일반화 성능을 유지합니다.

In [ ]:
!pip -q install "transformers>=4.43.2" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade

In [ ]:
import os, math, random
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any, List, Dict
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

# 하이퍼파라미터 설정
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 448 # 4050 메모리에 맞춰 조절 가능 (384-448 권장)
EPOCHS = 3
LR = 5e-5 # 과적합 방지를 위해 낮게 설정
GRAD_ACCUM = 16 # 4050 6GB 기준 배치를 키우기 위해 크게 설정
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
print(f"Total training data: {len(train_df)}")

In [ ]:
# 2. 모델 로드 및 LoRA 설정 (RTX 4050 최적화)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Qwen2.5-VL 전용 프로세서 설정
processor = AutoProcessor.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True,
    min_pixels=256*28*28,
    max_pixels=IMAGE_SIZE*28*28
)

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable() # 메모리 절약 핵심

lora_config = LoraConfig(
    r=32,           
    lora_alpha=64,  
    lora_dropout=0.1, # 과적합 방지 위해 상향
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 3. 정밀한 Masking을 위한 데이터셋
class VQAOptimizedDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        prompt = f"{row['question']}\n(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n정답:"
        answer = str(row["answer"]).strip().lower()

        # Chat Template 구성
        messages = [
            {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        ]
        return {"messages": messages, "image": img}

@dataclass
class OptimizedCollator:
    processor: Any
    def __call__(self, batch):
        texts = [self.processor.apply_chat_template(s["messages"], tokenize=False) for s in batch]
        images = [s["image"] for s in batch]
        
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        
        labels = enc["input_ids"].clone()
        
        # 정확한 Label Masking 로직
        for i in range(labels.shape[0]):
            # assistant의 시작 위치를 찾기 위해 템플릿 재구성
            full_text = texts[i]
            # Qwen2-VL assistant prefix
            assistant_prefix = "<|im_start|>assistant\n"
            user_part = full_text.split(assistant_prefix)[0] + assistant_prefix
            
            user_token_len = len(self.processor.tokenizer.encode(user_part))
            # assistant 응답 시작 전까지 모두 -100 처리
            labels[i, :user_token_len] = -100 
            
        enc["labels"] = labels
        return enc

train_ds = VQAOptimizedDataset(train_df, processor)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=OptimizedCollator(processor))

In [ ]:
# 4. 학습 실행
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05) # Regularization
num_training_steps = EPOCHS * (len(train_loader) // GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(num_training_steps*0.1), num_training_steps)
scaler = torch.cuda.amp.GradScaler()

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    running_loss = 0.0
    
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM
        
        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            pbar.set_postfix({"loss": f"{running_loss:.4f}"})
            running_loss = 0.0

model.save_pretrained("qwen_best_lora_optimized")
print("Training Complete!")

In [ ]:
# 5. Logit-based Inference (성능 극대화)
model.eval()
results = []

# 'a', 'b', 'c', 'd' 토큰 ID 미리 확인
option_tokens = ["a", "b", "c", "d"]
option_token_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in option_tokens]

for i in tqdm(range(len(test_df)), desc="Inference"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    prompt = f"{row['question']}\n(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n정답:"
    
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :] # 마지막 토큰의 Logit 추출
        
        # a, b, c, d 중 확률이 가장 높은 것 선택
        target_logits = logits[option_token_ids]
        best_idx = torch.argmax(target_logits).item()
        ans = option_tokens[best_idx]
    
    results.append(ans)

submission = pd.DataFrame({"id": test_df["id"], "answer": results})
submission.to_csv("submission_optimized_vqa.csv", index=False)
print("Submission file saved with logit-based inference!")